# GitHub Issues Dataset
### EDA, Data Cleaning & Feature Engineering

This dataset is an adaptation of the original [found on Kaggle](https://www.kaggle.com/datasets/tobiasbueck/helpdesk-github-tickets/data) that removes PII from the data prior to processing and model training. User IDs and username columns have been removed entirely, and @ mentions of usernames have been replace with the generic '@user` text.

## Import and Setup

In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# import plotly.express as px
# import plotly.graph_objects as go
# from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Text processing
import re
from collections import Counter
from wordcloud import WordCloud
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.sentiment import SentimentIntensityAnalyzer

# Date/time processing
from datetime import datetime, timedelta
import dateutil.parser as parser

# Feature engineering
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder, StandardScaler

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

# Load the dataset
github_df = pd.read_csv('github_issues.csv')

## Initial Exploration

In [25]:
github_df.head()

,created_at,repo_name,title,body,labels_0_color,labels_0_description,labels_0_name,labels_10_color,labels_10_description,labels_10_name,labels_1_color,labels_1_description,labels_1_name,labels_2_color,labels_2_description,labels_2_name,labels_3_color,labels_3_description,labels_3_name,labels_4_color,labels_4_description,labels_4_name,labels_5_color,labels_5_description,labels_5_name,labels_6_color,labels_6_description,labels_6_name,labels_7_color,labels_7_description,labels_7_name,labels_8_color,labels_8_description,labels_8_name,labels_9_color,labels_9_description,labels_9_name,answers_0_body,answers_0_creation_time,answers_1_body,answers_1_creation_time,answers_2_body,answers_2_creation_time,answers_3_body,answers_3_creation_time,answers_4_body,answers_4_creation_time,answers_5_body,answers_5_creation_time,answers_6_body,answers_6_creation_time,answers_7_body,answers_7_creation_time,answers_8_body,answers_8_creation_time,answers_9_body,answers_9_creation_time,closed_at
0,2023-05-05T15:54:28Z,angular/angular,Article mistake,### Describe the problem that you experienced\n\nI think there is a samll mistake in the tutoria...,8672,An issue that is suitable for a community contributor (based on its complexity/scope).,help wanted,NaN,NaN,NaN,7057ff,An issue that is suitable for first-time contributors; often a documentation issue.,good first issue,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Hey @user-agius4, can I work on this issue?",2023-05-05T18:09:31+00:00,@user You can open a PR if you'd like to fix that issue 😊,2023-05-05T18:19:32+00:00,"Following the history and looking at the page, it looks like this issue can be closed 😄 \r\n![Sc...",2023-05-16T21:51:45+00:00,This issue has been automatically locked due to inactivity.\nPlease file a new issue if you are ...,2023-06-19T00:09:04+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023-05-19T08:06:42Z
1,2024-06-07T20:52:24Z,microsoft/microsoft-ui-xaml,Able to change the window height even if IsResizable is false but ExtendsContentIntoTitleBar is ...,### Describe the bug\n\nIf you set ExtendsContentIntoTitleBar to true you'll still be able to ch...,d73a4a,Something isn't working,bug,NaN,NaN,NaN,7b03b2,"Issue for IXP (Composition, Input) team",team-CompInput,006b75,Issues related to custom window title bars.,area-TitleBar,D93F0B,NaN,Regression,006b75,NaN,area-Windowing,36ABD5,Described behavior has been fixed.,closed-Fixed,007F00,"The fix has been in a release (experimental, preview, stable, or servicing).",fix-released,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Hi I'm an AI powered bot that finds similar issues based off the issue title.\n\n Please view th...,2024-06-07T20:52:50+00:00,"This is very similar to #9670, though that issue is only talking about the cursor being incorrec...",2024-06-08T02:22:56+00:00,This was closed as fixed/duplicate of #7629. The fix will be in the next 1.6 release.,2024-07-11T02:02:16+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-07-10T07:06:59Z
2,2018-07-20T15:15:16Z,dotnet/roslyn,Introduce local for 'this' is not very useful,**Version Used**: VS 15.7\r\n\r\n**Steps to Reproduce**:\r\n\r\n1. Move caret to `this` and pres...,e52727,NaN,Bug,NaN,NaN,NaN,0e8a16,"The issue is ""up for grabs"" - add a comment if you are interested in working on it",help wanted,5.319E+10,NaN,Area-IDE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Why is this a bug?,2018-09-24T20:37:57+00:00,"Design Meeting Notes:\r\n\r\nSince `s` is `readonly`, it will cause an error. But more broadly, ...",2018-09-24T22:49:18+00:00,@user is there a good use case for this that we've missed?,2018-09-24T22:49:22+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-11-01T03:42:28Z
3,2019-09-09T19:47:02Z,rails/webpacker,bundle exec rails webpacker:install:typescript creates an invalid entry in config/webpack/enviro...,webpacker version: 4.0.7\r\n\r\nIt appends the line whi

In [26]:
print(f"Shape: {github_df.shape}")
print(f"Columns: {len(github_df.columns)}")

Shape: (15955, 58)
Columns: 58


### Data Types and Missing Values

In [27]:
info_df = pd.DataFrame({
    'Column': github_df.columns,
    'Data_Type': github_df.dtypes,
    'Non_Null_Count': github_df.count(),
    'Null_Count': github_df.isnull().sum(),
    'Null_Percentage': (github_df.isnull().sum() / len(github_df) * 100).round(2)
})
print(info_df.to_string(index=False))

                 Column Data_Type  Non_Null_Count  Null_Count  Null_Percentage
             created_at    object           15955           0             0.00
              repo_name    object           15955           0             0.00
                  title    object           15955           0             0.00
                   body    object           15955           0             0.00
         labels_0_color    object           15946           9             0.06
   labels_0_description    object            7263        8692            54.48
          labels_0_name    object           15946           9             0.06
        labels_10_color    object               4       15951            99.97
  labels_10_description   float64               0       15955           100.00
         labels_10_name    object               4       15951            99.97
         labels_1_color    object            8438        7517            47.11
   labels_1_description    object            3941   

## Cleaning

In [28]:
# Create a new copy
df = github_df.copy()

### Convert date columns to datetime

In [29]:
date_columns = ['created_at', 'closed_at']
answer_date_cols = [col for col in df.columns if 'answers_' in col and 'creation_time' in col]
date_columns.extend(answer_date_cols)

for col in date_columns:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

### Clean text

In [30]:
def clean_text(text):
    """Clean text by removing extra whitespace and handling NaN"""
    if pd.isna(text):
        return ""
    text = str(text)
    # Remove extra whitespace
    text = ' '.join(text.split())
    # Replace common escape characters
    text = text.replace('\\n', ' ').replace('\\t', ' ').replace('\\r', ' ')
    return text

text_columns = ['title', 'body', 'repo_name']
answers_text_cols = [col for col in df.columns if 'answers' in col and 'body' in col]
text_columns.extend(answers_text_cols)

for col in text_columns:
    if col in df.columns:
        df[col] = df[col].apply(clean_text)

### Label Processing

In [31]:
def extract_labels(row):
    """Extract all non-null labels from label columns, replacing NaN with empty strings"""
    labels = []
    for i in range(11):  # labels_0 to labels_10
        name_col = f'labels_{i}_name'
        color_col = f'labels_{i}_color'
        desc_col = f'labels_{i}_description'

        name = row.get(name_col, '')
        color = row.get(color_col, '')
        description = row.get(desc_col, '')

        # Replace NaN with empty string
        name = '' if pd.isna(name) else name
        color = '' if pd.isna(color) else color
        description = '' if pd.isna(description) else description

        if name:  # only include labels that have a name
            labels.append({
                'name': name,
                'color': color,
                'description': description
            })
    return labels

# Apply label extraction
df['labels'] = df.apply(extract_labels, axis=1)
df['n_labels'] = df['labels'].apply(len)

# Drop source columns
df = df.drop(columns=[col for col in df.columns if col.startswith('labels_')])

### Answers processing

In [32]:
answer_body_cols = [f'answers_{i}_body' for i in range(10)]
answer_time_cols = [f'answers_{i}_creation_time' for i in range(10)]

def merge_answers(row):
    answers = []
    for body_col, time_col in zip(answer_body_cols, answer_time_cols):
        body = row.get(body_col)
        time = row.get(time_col)
        if pd.notna(body) and pd.notna(time):
            answers.append({
                "timestamp": str(time),
                "body": str(body)
            })
    return answers

df['answers'] = df.apply(merge_answers, axis=1)
df['n_answers'] = df.apply(lambda x: len(x['answers']), axis=1)
df = df.drop(columns=[col for col in df.columns if col.startswith('answers_')])
df.head(3)

,created_at,repo_name,title,body,closed_at,labels,n_labels,answers,n_answers
0,2023-05-05 15:54:28+00:00,angular/angular,Article mistake,### Describe the problem that you experienced I think there is a samll mistake in the tutorial i...,2023-05-19 08:06:42+00:00,"[{'name': 'help wanted', 'color': '8672', 'description': 'An issue that is suitable for a commun...",2,"[{'timestamp': '2023-05-05 18:09:31+00:00', 'body': 'Hey @user-agius4, can I work on this issue?...",4
1,2024-06-07 20:52:24+00:00,microsoft/microsoft-ui-xaml,Able to change the window height even if IsResizable is false but ExtendsContentIntoTitleBar is ...,### Describe the bug If you set ExtendsContentIntoTitleBar to true you'll still be able to chang...,2024-07-10 07:06:59+00:00,"[{'name': 'bug', 'color': 'd73a4a', 'description': 'Something isn't working'}, {'name': 'team-Co...",7,"[{'timestamp': '2024-06-07 20:52:50+00:00', 'body': 'Hi I'm an AI powered bot that finds similar...",3
2,2018-07-20 15:15:16+00:00,dotnet/roslyn,Introduce local for 'this' is not very useful,**Version Used**: VS 15.7 **Steps to Reproduce**: 1. Move caret to `this` and press `Ctrl+.` ```...,2022-11-01 03:42:28+00:00,"[{'name': 'Bug', 'color': 'e52727', 'description': ''}, {'name': 'help wanted', 'color': '0e8a16...",3,"[{'timestamp': '2018-09-24 20:37:57+00:00', 'body': 'Why is this a bug?'}, {'timestamp': '2018-0...",3


## Feature Engineering

### Time Features

In [33]:
# Extract time features
df['created_year'] = df['created_at'].dt.year
df['created_month'] = df['created_at'].dt.month
df['created_day_of_week'] = df['created_at'].dt.dayofweek
df['created_hour'] = df['created_at'].dt.hour

df['closed_year'] = df['closed_at'].dt.year
df['closed_month'] = df['closed_at'].dt.month
df['closed_day_of_week'] = df['closed_at'].dt.dayofweek
df['closed_hour'] = df['closed_at'].dt.hour
df['resolution_time'] = round((df['closed_at'] - df['created_at']).dt.total_seconds() / 3600, 2)

# Drop source columns
df.drop(columns=['created_at', 'closed_at'], axis=1, inplace=True)
df.head(3)

,repo_name,title,body,labels,n_labels,answers,n_answers,created_year,created_month,created_day_of_week,created_hour,closed_year,closed_month,closed_day_of_week,closed_hour,resolution_time
0,angular/angular,Article mistake,### Describe the problem that you experienced I think there is a samll mistake in the tutorial i...,"[{'name': 'help wanted', 'color': '8672', 'description': 'An issue that is suitable for a commun...",2,"[{'timestamp': '2023-05-05 18:09:31+00:00', 'body': 'Hey @user-agius4, can I work on this issue?...",4,2023,5,4,15,2023,5,4,8,328.20
1,microsoft/microsoft-ui-xaml,Able to change the window height even if IsResizable is false but ExtendsContentIntoTitleBar is ...,### Describe the bug If you set ExtendsContentIntoTitleBar to true you'll still be able to chang...,"[{'name': 'bug', 'color': 'd73a4a', 'description': 'Something isn't working'}, {'name': 'team-Co...",7,"[{'timestamp': '2024-06-07 20:52:50+00:00', 'body': 'Hi I'm an AI powered bot that finds similar...",3,2024,6,4,20,2024,7,2,7,778.24
2,dotnet/roslyn,Introduce local for 'this' is not very useful,**Version Used**: VS 15.7 **Steps to Reproduce**: 1. Move caret to `this` and press `Ctrl+.` ```...,"[{'name': 'Bug', 'color': 'e52727', 'description': ''}, {'name': 'help wanted', 'color': '0e8a16...",3,"[{'timestamp': '2018-09-24 20:37:57+00:00', 'body': 'Why is this a bug?'}, {'timestamp': '2018-0...",3,2018,7,4,15,2022,11,1,3,37548.45


repo_name              0
title                  0
body                   0
labels                 0
n_labels               0
answers                0
n_answers              0
created_year           0
created_month          0
created_day_of_week    0
created_hour           0
closed_year            0
closed_month           0
closed_day_of_week     0
closed_hour            0
resolution_time        0
dtype: int64

### Text Features

In [37]:
# Calculate text lengths
df['title_length'] = df['title'].str.len()
df['body_length'] = df['body'].str.len()
df['title_word_count'] = df['title'].str.split().str.len()
df['body_word_count'] = df['body'].str.split().str.len()

In [39]:
def extract_text_features(data_frame):
    """Extract various text-based features"""

    # Basic text features
    data_frame['has_empty_body'] = (data_frame['body_length'] == 0).astype(int)

    # Code-related features
    data_frame['has_code_blocks'] = data_frame['body'].str.contains('```', na=False).astype(int)
    data_frame['code_block_count'] = data_frame['body'].str.count('```') // 2

    # URL features
    data_frame['url_count'] = data_frame['body'].str.count(r'https?://[^\s]+')
    data_frame['has_urls'] = (data_frame['url_count'] > 0).astype(int)

    # Question indicators
    question_words = ['how', 'what', 'why', 'when', 'where', 'which', 'who']
    data_frame['question_word_count'] = data_frame['title'].str.lower().str.count('|'.join(question_words))
    data_frame['has_question_mark'] = data_frame['title'].str.contains('\?', na=False).astype(int)
    data_frame['includes_questions'] = ((data_frame['question_word_count'] > 0) | (data_frame['has_question_mark'] > 0)).astype(int)

    # Urgency indicators
    urgent_words = ['urgent', 'critical', 'asap', 'immediate', 'emergency', 'broken', 'error', 'serious', 'security']
    data_frame['n_urgent_words'] = data_frame['title'].str.lower().str.count('|'.join(urgent_words))
    data_frame['has_exclamation'] = data_frame['title'].str.contains('!', na=False).astype(int)

    return data_frame

df = extract_text_features(df)

In [40]:
df.head(3)

,repo_name,title,body,labels,n_labels,answers,n_answers,created_year,created_month,created_day_of_week,created_hour,closed_year,closed_month,closed_day_of_week,closed_hour,resolution_time,title_length,body_length,title_word_count,body_word_count,has_empty_body,has_code_blocks,code_block_count,url_count,has_urls,question_word_count,has_question_mark,includes_questions,n_urgent_words,has_exclamation
0,angular/angular,Article mistake,### Describe the problem that you experienced I think there is a samll mistake in the tutorial i...,"[{'name': 'help wanted', 'color': '8672', 'description': 'An issue that is suitable for a commun...",2,"[{'timestamp': '2023-05-05 18:09:31+00:00', 'body': 'Hey @user-agius4, can I work on this issue?...",4,2023,5,4,15,2023,5,4,8,328.20,15,1107,2,166,0,0,0,2,1,0,0,0,0,0
1,microsoft/microsoft-ui-xaml,Able to change the window height even if IsResizable is false but ExtendsContentIntoTitleBar is ...,### Describe the bug If you set ExtendsContentIntoTitleBar to true you'll still be able to chang...,"[{'name': 'bug', 'color': 'd73a4a', 'description': 'Something isn't working'}, {'name': 'team-Co...",7,"[{'timestamp': '2024-06-07 20:52:50+00:00', 'body': 'Hi I'm an AI powered bot that finds similar...",3,2024,6,4,20,2024,7,2,7,778.24,100,618,15,94,0,0,0,0,0,0,0,0,0,0
2,dotnet/roslyn,Introduce local for 'this' is not very useful,**Version Used**: VS 15.7 **Steps to Reproduce**: 1. Move caret to `this` and press `Ctrl+.` ```...,"[{'name': 'Bug', 'color': 'e52727', 'description': ''}, {'name': 'help wanted', 'color': '0e8a16...",3,"[{'timestamp': '2018-09-24 20:37:57+00:00', 'body': 'Why is this a bug?'}, {'timestamp': '2018-0...",3,2018,7,4,15,2022,11,1,3,37548.45,45,511,8,88,0,1,1,0,0,0,0,0,0,0


### Label Features

In [45]:
def extract_label_features(data_frame):
    """Extract label-based features"""

    # Label category features
    bug_labels = ['bug', 'error', 'issue', 'problem', 'broken']
    feature_labels = ['feature', 'enhancement', 'request', 'improvement']
    doc_labels = ['documentation', 'docs', 'readme']
    help_labels = ['help', 'question', 'support']

    def has_label_category(labels, category_words):
        return any(any(word in label["name"] for word in category_words) for label in labels)

    data_frame['has_bug_label'] = data_frame['labels'].apply(lambda x: has_label_category(x, bug_labels)).astype(int)
    data_frame['has_feature_label'] = data_frame['labels'].apply(lambda x: has_label_category(x, feature_labels)).astype(int)
    data_frame['has_doc_label'] = data_frame['labels'].apply(lambda x: has_label_category(x, doc_labels)).astype(int)
    data_frame['has_help_label'] = data_frame['labels'].apply(lambda x: has_label_category(x, help_labels)).astype(int)

    return data_frame

df = extract_label_features(df)

In [47]:
df.head(3)

,repo_name,title,body,labels,n_labels,answers,n_answers,created_year,created_month,created_day_of_week,created_hour,closed_year,closed_month,closed_day_of_week,closed_hour,resolution_time,title_length,body_length,title_word_count,body_word_count,has_empty_body,has_code_blocks,code_block_count,url_count,has_urls,question_word_count,has_question_mark,includes_questions,n_urgent_words,has_exclamation,has_bug_label,has_feature_label,has_doc_label,has_help_label
0,angular/angular,Article mistake,### Describe the problem that you experienced I think there is a samll mistake in the tutorial i...,"[{'name': 'help wanted', 'color': '8672', 'description': 'An issue that is suitable for a commun...",2,"[{'timestamp': '2023-05-05 18:09:31+00:00', 'body': 'Hey @user-agius4, can I work on this issue?...",4,2023,5,4,15,2023,5,4,8,328.20,15,1107,2,166,0,0,0,2,1,0,0,0,0,0,1,0,0,1
1,microsoft/microsoft-ui-xaml,Able to change the window height even if IsResizable is false but ExtendsContentIntoTitleBar is ...,### Describe the bug If you set ExtendsContentIntoTitleBar to true you'll still be able to chang...,"[{'name': 'bug', 'color': 'd73a4a', 'description': 'Something isn't working'}, {'name': 'team-Co...",7,"[{'timestamp': '2024-06-07 20:52:50+00:00', 'body': 'Hi I'm an AI powered bot that finds similar...",3,2024,6,4,20,2024,7,2,7,778.24,100,618,15,94,0,0,0,0,0,0,0,0,0,0,1,0,0,0
2,dotnet/roslyn,Introduce local for 'this' is not very useful,**Version Used**: VS 15.7 **Steps to Reproduce**: 1. Move caret to `this` and press `Ctrl+.` ```...,"[{'name': 'Bug', 'color': 'e52727', 'description': ''}, {'name': 'help wanted', 'color': '0e8a16...",3,"[{'timestamp': '2018-09-24 20:37:57+00:00', 'body': 'Why is this a bug?'}, {'timestamp': '2018-0...",3,2018,7,4,15,2022,11,1,3,37548.45,45,511,8,88,0,1,1,0,0,0,0,0,0,0,0,0,0,1
